<a href="https://colab.research.google.com/github/gershonc/Ollama-Ngrok-Colab/blob/main/Running_ollama_in_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Free Ollama Colab Deploy

Welcome to the easiest way to deploy local Large Language Models (LLMs) for free!

This notebook demonstrates how to install and run **Ollama** directly within a Google Colab instance, and then securely expose the API using **ngrok**. This gives you a public API endpoint to interact with models like `qwen:0.5b`, `llama3`, and others from your local machine, applications, or anywhere in the world—completely free of charge.

### ✨ Features:
* **Zero-Cost Deployment:** Utilize Google Colab's free computing power.
* **Quick Setup:** Install and run Ollama in seconds.
* **Public API:** Seamlessly connect to your model via an ngrok public URL.

### 🛠️ Getting Started:
1. Run the setup cells to install Ollama.
2. Add your `NGROK_AUTH_TOKEN` in the Colab secrets tab (🔑).
3. Run the ngrok cell to get your public API link!

Let's get started! 👇

### 1. Install Ollama

First, we need to download and install Ollama. This will set up the necessary binaries on your Colab instance.

In [ ]:
%%bash
sudo apt-get update
sudo apt-get install -y zstd
curl -fsSL https://ollama.com/install.sh | sh

### 2. Start the Ollama server

Ollama runs as a server process. We need to start it in the background so it can serve models.

In [ ]:
import subprocess
import os
import time
import urllib.request
import urllib.error

# Set the OLLAMA_HOST environment variable to allow access from outside the server
os.environ['OLLAMA_HOST'] = '0.0.0.0'

# Start Ollama server in the background
# Use 'nohup' to keep it running even if the current shell exits
# Redirect output to a log file
ollama_process = subprocess.Popen(['nohup', 'ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, preexec_fn=os.setpgrp)

print("Ollama server started in the background.")
print("Waiting for the server to initialize...")

# Actively check if the server is ready
max_retries = 30
for i in range(max_retries):
    try:
        response = urllib.request.urlopen('http://localhost:11434/')
        if response.getcode() == 200:
            print("\nOllama server is ready!")
            break
    except urllib.error.URLError:
        print(".", end="", flush=True)
    time.sleep(1)
else:
    print("\nWarning: Ollama server did not respond in time. It might have failed to start.")

### 3. Pre-pull a recommended tiny model (e.g., `qwen:0.5b`)

This step downloads the model to your Colab instance, making subsequent uses faster as it won't need to download it again during `ollama run`.

In [13]:
%%bash
/usr/local/bin/ollama pull qwen:0.5b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling fad2a06e4cc7: 100% ▕██████████████████▏ 394 MB                         
pulling 41c2cf8c272f: 100% ▕██████████████████▏ 7.3 KB                         
pulling 1da0581fd4ce: 100% ▕██████████████████▏  130 B                         
pulling f02dd72bb242: 100% ▕██████████████████▏   59 B                         
pulling ea0a531a015b: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


### 4. Run the `qwen:0.5b` model

Now you can run a prompt using the downloaded `qwen:0.5b` model. If you want to use `llama2` or another model, just replace `qwen:0.5b` with its name.

In [14]:
%%bash
/usr/local/bin/ollama run qwen:0.5b "What is the capital of France? Answer in one word."

Paris.



⠙ ⠹ ⠸ ⠼ 

In [15]:
%%bash
ollama run qwen:0.5b "Why is the sky blue? Answer in one short sentence."

The reason why the sky is blue is that it takes up a significant amount of 
the Earth's solar system, which causes the sun to be heated and absorbed by
by space.

As the sun warms and absorbs space, the temperature inside the sun changes.
changes. This leads to a process called convection, where heat is transferr
transferred from the hot surface to the cold surface in order to transfer热
量 between different regions on the planet.

In summary, the sky blue is due to the fact that it takes up a significant 
amount of the Earth's solar system, which causes the sun to be heated and a
absorbed by space.



⠙ ⠹ 

### 5. Expose Ollama to your local PC using ngrok

To access the Ollama server from your local machine, we need to create a public tunnel to the Colab instance. We'll use `ngrok` for this.

**Note**: You'll need an `ngrok` account and an authtoken. You can get one from [ngrok's website](https://ngrok.com/). Add your authtoken to Colab's secrets manager (the '🔑' icon in the left panel) with the name `NGROK_AUTH_TOKEN`.

In [ ]:
print('Installing ngrok...')
!pip install pyngrok --quiet

import os
from pyngrok import ngrok, conf
import time
from google.colab import userdata

# Get ngrok authtoken from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

if NGROK_AUTH_TOKEN is None:
    raise ValueError("NGROK_AUTH_TOKEN not found in Colab secrets. Please add it.")

# Authenticate ngrok
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Ensure any previous ngrok processes are killed
!killall ngrok > /dev/null 2>&1 || true # Suppress output and ignore if no processes are found
ngrok.kill() # Kill pyngrok-managed tunnels
time.sleep(2) # Give ngrok a moment to shut down and release resources

# Disconnect any lingering tunnels reported by pyngrok
try:
    for tunnel in ngrok.get_tunnels():
        print(f"Disconnecting existing tunnel: {tunnel.public_url}")
        ngrok.disconnect(tunnel.public_url)
    print("Finished disconnecting existing ngrok tunnels.")
except Exception as e:
    print(f"Error during ngrok tunnel disconnection: {e}")

time.sleep(2) # Additional pause after disconnection

# Open a tunnel to the Ollama port (11434)
print('Starting ngrok tunnel...')
public_url = ngrok.connect(11434)

print(f"Ollama public URL: {public_url}")
print("You can now access Ollama from your local machine using this URL.")
print("Example: `ollama run llama2 --host {public_url}` on your local terminal")

In [25]:
import requests
import json

# Dynamically use the public_url from the ngrok cell (extracting the string from the object)
url = f"{public_url.public_url}/api/generate"

payload = {
    "model": "qwen:0.5b",
    "prompt": "What is the fastest land animal?",
    "stream": False
}

print(f"🚀 Sending request to: {url}...")

try:
    response = requests.post(url, json=payload)
    response.raise_for_status() # Check for any HTTP errors

    # Parse the JSON and extract the text response
    result = response.json()
    print("\n✨ Model Response ✨")
    print("-" * 40)
    print(result.get('response', 'No response field found in JSON.'))
    print("-" * 40)

    # Remove the context array to avoid cluttering the output
    result.pop('context', None)

    print("\n📊 Full Metadata (excluding context) 📊")
    print("-" * 40)
    print(json.dumps(result, indent=2))
    print("-" * 40)

except requests.exceptions.RequestException as e:
    print(f"❌ Error connecting to the API: {e}")

🚀 Sending request to: https://boss-refurnish-agency.ngrok-free.dev/api/generate...

✨ Model Response ✨
----------------------------------------
The fastest land animal is the gorilla. Gorillas are known for their incredible speed and agility, as well as their unique physical form and habitat preferences.
----------------------------------------

📊 Full Metadata (excluding context) 📊
----------------------------------------
{
  "model": "qwen:0.5b",
  "created_at": "2026-04-15T09:36:47.267420982Z",
  "response": "The fastest land animal is the gorilla. Gorillas are known for their incredible speed and agility, as well as their unique physical form and habitat preferences.",
  "done": true,
  "done_reason": "stop",
  "total_duration": 468563377,
  "load_duration": 186397537,
  "prompt_eval_count": 15,
  "prompt_eval_duration": 6265252,
  "eval_count": 32,
  "eval_duration": 190135364
}
----------------------------------------


### 6. (Optional) Cleanup

*(Note: Make sure to run this only after you are done testing the API via ngrok!)*

If you want to stop the Ollama server, you can kill the process using the process ID.

In [ ]:
# To stop the Ollama server, you can kill the process.
# Be careful with killing processes, ensure it's the correct one.
# The 'ollama_process' variable holds the Popen object.
if 'ollama_process' in locals() and ollama_process.poll() is None:
    ollama_process.terminate() # or .kill() for a more forceful termination
    print("Ollama server terminated.")
else:
    print("Ollama server not found or already stopped.")